In [1]:
# ============================================================
# CELL 1: Setup
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, torch

class Config:
    PROJECT_ROOT  = '/content/drive/MyDrive/FinDocVQA'
    DATA_DIR      = os.path.join(PROJECT_ROOT, 'data')
    CHARTQA_DIR   = os.path.join(DATA_DIR, 'chartqa')
    MODELS_DIR    = os.path.join(PROJECT_ROOT, 'models')
    OUTPUTS_DIR   = os.path.join(PROJECT_ROOT, 'outputs')
    DEPLOT_ID     = 'google/deplot'
    SEED = 42

cfg = Config()
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
✓ GPU: Tesla T4


In [3]:
# ============================================================
# CELL 2: Install + Load ChartQA
# ============================================================

%%capture
!pip install transformers accelerate datasets sentencepiece protobuf

from datasets import load_from_disk
import numpy as np

chartqa = load_from_disk(os.path.join(cfg.CHARTQA_DIR, 'chartqa_dataset'))
print(f"✓ ChartQA loaded: {len(chartqa['val'])} test examples")

In [4]:
# ============================================================
# CELL 3: Load DePlot + Build Two-Stage Pipeline
# ============================================================
# Stage 1: DePlot converts chart image → linearized table
# Stage 2: We use a FLAN-T5 model to reason over the table
#           and answer the question
# ============================================================

from transformers import (
    Pix2StructProcessor,
    Pix2StructForConditionalGeneration,
    T5Tokenizer,
    T5ForConditionalGeneration
)
from PIL import Image
from io import BytesIO

# --- Stage 1: DePlot (chart → table) ---
print("Loading DePlot...")
deplot_processor = Pix2StructProcessor.from_pretrained(cfg.DEPLOT_ID)
deplot_model = Pix2StructForConditionalGeneration.from_pretrained(cfg.DEPLOT_ID)
deplot_model = deplot_model.to('cuda')
deplot_model.eval()
print(f"✓ DePlot loaded ({sum(p.numel() for p in deplot_model.parameters()) / 1e6:.1f}M params)")

# --- Stage 2: FLAN-T5 (table + question → answer) ---
print("Loading FLAN-T5-base...")
FLAN_ID = 'google/flan-t5-base'
flan_tokenizer = T5Tokenizer.from_pretrained(FLAN_ID)
flan_model = T5ForConditionalGeneration.from_pretrained(FLAN_ID)
flan_model = flan_model.to('cuda')
flan_model.eval()
print(f"✓ FLAN-T5 loaded ({sum(p.numel() for p in flan_model.parameters()) / 1e6:.1f}M params)")

def deplot_llm_predict(image, question):
    """
    Two-stage pipeline:
    1. DePlot: chart image → linearized table
    2. FLAN-T5: table + question → answer
    """
    if isinstance(image, dict) and 'bytes' in image:
        image = Image.open(BytesIO(image['bytes']))
    image = image.convert('RGB')

    # Stage 1: Extract table from chart
    inputs = deplot_processor(
        images=image,
        text="Generate underlying data table of the figure below:",
        return_tensors="pt",
        max_patches=2048
    ).to('cuda')

    with torch.no_grad():
        generated = deplot_model.generate(**inputs, max_new_tokens=512)
    table_text = deplot_processor.decode(generated[0], skip_special_tokens=True)

    # Stage 2: Reason over the table
    prompt = (
        f"Answer the following question based on the data table.\n\n"
        f"Table:\n{table_text}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )

    flan_inputs = flan_tokenizer(
        prompt, return_tensors="pt", max_length=512, truncation=True
    ).to('cuda')

    with torch.no_grad():
        flan_output = flan_model.generate(
            **flan_inputs,
            max_new_tokens=64
        )
    answer = flan_tokenizer.decode(flan_output[0], skip_special_tokens=True).strip()

    return {"answer": answer, "table": table_text}

# --- Quick test ---
example = chartqa['test'][0]
print(f"\nTest example:")
print(f"  Question: {example['query']}")
print(f"  Ground truth: {example['label']}")

result = deplot_llm_predict(example['image'], example['query'])
print(f"  Extracted table: {result['table'][:200]}...")
print(f"  Predicted: {result['answer']}")

Loading DePlot...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

The image processor of type `Pix2StructImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✓ FLAN-T5 loaded (247.6M params)

Test example:
  Question: How many food item is shown in the bar graph?
  Ground truth: ['14']


Arial.TTF: 0.00B [00:00, ?B/s]

  Extracted table: Country | Long-term price index in food commodities, 1850-2015, World, 1934 <0x0A> Lamb | 103.7 <0x0A> Corn | 103.13 <0x0A> Barley | 102.46 <0x0A> Rye | 87.37 <0x0A> Beef | 85.27 <0x0A> Wheat | 83.73 ...
  Predicted: 0


In [5]:
# ============================================================
# CELL 4: Evaluate DePlot+LLM on ChartQA Validation
# ============================================================

from tqdm import tqdm
import time
import json

def normalized_levenshtein(s1, s2):
    s1, s2 = s1.lower().strip(), s2.lower().strip()
    if s1 == s2:
        return 1.0
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    dist = dp[m][n]
    return 1.0 - (dist / max(m, n))

def compute_anls(prediction, ground_truths, threshold=0.5):
    if not prediction:
        return 0.0
    best = max(normalized_levenshtein(prediction, gt) for gt in ground_truths)
    return best if best >= threshold else 0.0

def compute_exact_match(prediction, ground_truths):
    pred = prediction.lower().strip()
    return float(any(pred == gt.lower().strip() for gt in ground_truths))

# --- Evaluate on ChartQA val ---
NUM_EVAL = 500
val_data = chartqa['val']
np.random.seed(42)
eval_indices = np.random.choice(len(val_data), NUM_EVAL, replace=False)

results = []
anls_scores = []
em_scores = []

print(f"Evaluating DePlot+FLAN-T5 on {NUM_EVAL} ChartQA val examples...")
print("=" * 60)

start_time = time.time()

for i, idx in enumerate(tqdm(eval_indices, desc="Evaluating")):
    example = val_data[int(idx)]
    question = example['query']
    ground_truths = example['label']

    try:
        result = deplot_llm_predict(example['image'], question)
        prediction = result['answer']
        table = result['table']
    except Exception as e:
        print(f"  Error on idx {idx}: {e}")
        prediction = ""
        table = ""

    anls = compute_anls(prediction, ground_truths)
    em = compute_exact_match(prediction, ground_truths)

    anls_scores.append(anls)
    em_scores.append(em)

    results.append({
        'index': int(idx),
        'question': question,
        'ground_truths': ground_truths,
        'prediction': prediction,
        'extracted_table': table[:300],  # truncate for storage
        'anls': anls,
        'exact_match': em,
    })

elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"DEPLOT + FLAN-T5 RESULTS ({NUM_EVAL} examples)")
print("=" * 60)
print(f"  ANLS:         {np.mean(anls_scores):.4f}")
print(f"  Exact Match:  {np.mean(em_scores):.4f}")
print(f"  Time:         {elapsed:.0f}s ({elapsed/NUM_EVAL:.1f}s per example)")
print("=" * 60)

# Save results
results_path = os.path.join(cfg.OUTPUTS_DIR, 'deplot_flan_chartqa_results.json')
with open(results_path, 'w') as f:
    json.dump({
        'model': 'DePlot + FLAN-T5-base (two-stage pipeline)',
        'dataset': 'ChartQA validation',
        'num_examples': NUM_EVAL,
        'anls': float(np.mean(anls_scores)),
        'exact_match': float(np.mean(em_scores)),
        'per_example': results
    }, f, indent=2)
print(f"\n✓ Results saved to {results_path}")

Evaluating DePlot+FLAN-T5 on 500 ChartQA val examples...


Evaluating: 100%|██████████| 500/500 [1:05:38<00:00,  7.88s/it]


DEPLOT + FLAN-T5 RESULTS (500 examples)
  ANLS:         0.4413
  Exact Match:  0.2940
  Time:         3938s (7.9s per example)

✓ Results saved to /content/drive/MyDrive/FinDocVQA/outputs/deplot_flan_chartqa_results.json


In [6]:
# ============================================================
# CELL 5: Compare — Pix2Struct on same ChartQA examples
# ============================================================
# Pix2Struct-DocVQA is an end-to-end model. Let's see how it
# handles chart questions compared to the DePlot pipeline.
# ============================================================

from transformers import Pix2StructProcessor, Pix2StructForConditionalGeneration

# Load Pix2Struct (DocVQA version)
print("Loading Pix2Struct for comparison...")
p2s_processor = Pix2StructProcessor.from_pretrained('google/pix2struct-docvqa-base')

# Free DePlot + FLAN from GPU to make room
deplot_model.cpu()
flan_model.cpu()
torch.cuda.empty_cache()

p2s_model = Pix2StructForConditionalGeneration.from_pretrained('google/pix2struct-docvqa-base')
p2s_model = p2s_model.to('cuda')
p2s_model.eval()
print("✓ Pix2Struct loaded")

def pix2struct_predict(image, question):
    if isinstance(image, dict) and 'bytes' in image:
        image = Image.open(BytesIO(image['bytes']))
    image = image.convert('RGB')
    inputs = p2s_processor(images=image, text=question, return_tensors="pt", max_patches=2048).to('cuda')
    with torch.no_grad():
        generated = p2s_model.generate(**inputs, max_new_tokens=256)
    return p2s_processor.decode(generated[0], skip_special_tokens=True)

# --- Evaluate on same indices ---
p2s_anls = []
p2s_em = []

print(f"\nEvaluating Pix2Struct on same {NUM_EVAL} ChartQA val examples...")

for idx in tqdm(eval_indices, desc="Pix2Struct"):
    example = val_data[int(idx)]
    question = example['query']
    ground_truths = example['label']

    try:
        prediction = pix2struct_predict(example['image'], question)
    except Exception as e:
        print(f"  Error: {e}")
        prediction = ""

    p2s_anls.append(compute_anls(prediction, ground_truths))
    p2s_em.append(compute_exact_match(prediction, ground_truths))

print("\n" + "=" * 60)
print("CHARTQA COMPARISON")
print("=" * 60)
print(f"  DePlot+FLAN-T5:  ANLS={np.mean(anls_scores):.4f}  EM={np.mean(em_scores):.4f}")
print(f"  Pix2Struct:      ANLS={np.mean(p2s_anls):.4f}  EM={np.mean(p2s_em):.4f}")
print("=" * 60)

Loading Pix2Struct for comparison...


preprocessor_config.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/851k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

✓ Pix2Struct loaded

Evaluating Pix2Struct on same 500 ChartQA val examples...


Pix2Struct: 100%|██████████| 500/500 [06:35<00:00,  1.26it/s]


CHARTQA COMPARISON
  DePlot+FLAN-T5:  ANLS=0.4413  EM=0.2940
  Pix2Struct:      ANLS=0.3139  EM=0.1880
